# SMS.AC LLM 2 — fresh detailed SDLC training

This notebook starts from Qwen3-4B-Instruct-2507 and trains **only** the SDLC adapter on the 1,008-record detailed v3 dataset. The old LLM 2 adapter and dataset are not loaded. Select a Kaggle GPU and enable Internet. Outputs remain synthetic-data research results until separately reviewed on real design documents.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys, zipfile

assert platform.system() == "Linux" and Path("/kaggle/working").is_dir(), "Run this notebook on Kaggle."
assert any(os.environ.get(k) for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_URL_BASE", "KAGGLE_DATA_PROXY_TOKEN")), "Kaggle environment marker is missing."

PROJECT = Path("/kaggle/working/SEproject")
RUN_FULL = True           # True trains every eligible training row for FULL_EPOCHS.
PILOT_STEPS = 150          # <=5 is recorded as smoke; >5 is pilot.
FULL_EPOCHS = 1
MAX_SEQ_LENGTH = 4096      # Complete prompt + answer; oversized rows fail, never truncate.
GRADIENT_ACCUMULATION = 8
RESUME = {"sdlc": None}  # "auto" or checkpoint directory after an interrupted run.
TEST_LIMIT = 24            # Set 0 for all held-out examples.
GENERATION_LIMIT = 3
CHALLENGE_LIMIT = 0        # Independent prompts, not expert-validated gold answers.
MAX_NEW_TOKENS = 3072
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BASE_REVISION = "main"     # Each run resolves this to an immutable commit and records it.

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["SMS_AC_GPU"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"  # Keep subprocess metrics visible during long runs.
os.environ["HF_HOME"] = "/tmp/sms-ac-huggingface"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
print("Run tier:", "full" if RUN_FULL else ("smoke" if PILOT_STEPS <= 5 else "pilot"))


## Load the private project package


In [ ]:
# Kaggle's browser uploader may expand ZIPs into an SEproject directory.
if not (PROJECT / "scripts/train_dual_qlora.py").exists():
    unpacked = [p.parent.parent for p in Path("/kaggle/input").rglob("scripts/train_dual_qlora.py")]
    if len(unpacked) > 1:
        raise RuntimeError(f"Attach exactly one project dataset: {unpacked}")
    if unpacked:
        import shutil
        shutil.copytree(unpacked[0], PROJECT, dirs_exist_ok=True)
if not (PROJECT / "scripts/train_dual_qlora.py").exists():
    accepted = {"project.zip", "seproject.zip", "sms-ac-project.zip"}
    archives = [p for p in Path("/kaggle/input").rglob("*.zip") if p.name.lower() in accepted]
    if len(archives) != 1:
        raise RuntimeError(f"Attach exactly one project.zip dataset. Candidates: {archives}")
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        entries = archive.infolist()
        if sum(item.file_size for item in entries) > 3 * 1024**3:
            raise ValueError("Project archive exceeds the 3 GiB extraction budget.")
        mapped = []
        seen = set()
        for item in entries:
            normalized = item.filename.replace("\\", "/")
            parts = Path(normalized).parts
            if parts and parts[0].lower() == "seproject":
                parts = parts[1:]
            if not parts:
                continue
            if normalized.startswith("/") or ".." in parts or any(":" in part for part in parts):
                raise ValueError(f"Unsafe archive path: {item.filename}")
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Archive symlinks are not allowed.")
            target = PROJECT.joinpath(*parts).resolve()
            if not target.is_relative_to(PROJECT.resolve()) or target in seen:
                raise ValueError(f"Unsafe or duplicate target: {target}")
            seen.add(target)
            mapped.append((item, target))
        for item, target in mapped:
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, target.open("wb") as destination:
                    import shutil
                    shutil.copyfileobj(source, destination)
assert (PROJECT / "requirements-kaggle.txt").exists()
for split in ("train", "validation", "test"):
    assert (PROJECT / "data/llm2_training/sdlc" / f"{split}.jsonl").is_file(), f"Missing detailed LLM2/{split}"
print("Project:", PROJECT)


## Install isolated Kaggle training environment


In [ ]:
import venv
VENV = Path("/tmp/sms-ac-training-venv")
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False).create(VENV)
# Kaggle may omit ensurepip; host pip can bootstrap an empty venv.
subprocess.run([sys.executable, "-m", "pip", "--python", str(PYTHON),
                "install", "pip==25.3"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT / "requirements-kaggle.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
(PROJECT / "reports").mkdir(exist_ok=True)
lock = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
(PROJECT / "reports/kaggle-environment.txt").write_text(lock)
print("Isolated training environment ready.")


## Validate all examples and full token lengths before training


In [ ]:
def run_script(name, *arguments):
    command = [str(PYTHON), str(PROJECT / "scripts" / name), *map(str, arguments)]
    subprocess.run(command, cwd=str(PROJECT), env=os.environ.copy(), check=True)

run_script("train_dual_qlora.py", "--task", "sdlc", "--data-dir", PROJECT / "data/llm2_training",
           "--model", BASE_MODEL, "--revision", BASE_REVISION, "--max-seq-length", MAX_SEQ_LENGTH,
           "--output-dir", PROJECT / "models/preflight/llm2-detailed", "--preflight-only")


## Train fresh LLM 2 adapter for one complete epoch

The run uses only the new detailed dataset. It writes checkpoints and a manifest containing dataset hashes.


In [ ]:
arguments = [
    "--task", "sdlc", "--data-dir", PROJECT / "data/llm2_training",
    "--model", BASE_MODEL, "--revision", BASE_REVISION,
    "--max-seq-length", MAX_SEQ_LENGTH, "--gradient-accumulation", GRADIENT_ACCUMULATION,
    "--output-dir", PROJECT / "models/llm2-detailed-v3-adapter",
    "--save-steps", 25, "--eval-samples", 24,
]
arguments += ["--full", "--epochs", FULL_EPOCHS] if RUN_FULL else ["--max-steps", PILOT_STEPS]
if RESUME["sdlc"]:
    arguments += ["--resume-from-checkpoint", RESUME["sdlc"]]
run_script("train_dual_qlora.py", *arguments)
manifest = json.loads((PROJECT / "models/llm2-detailed-v3-adapter/model_manifest.json").read_text())
assert manifest["training_completed"] and manifest["full_dataset_epochs_completed"]
print("Completed:", manifest["global_step"], "steps;", manifest["trained_epochs"], "epochs")


## Compare base and adapter on held-out project families


In [ ]:
run_script("evaluate_dual_models.py", "--task", "sdlc",
           "--data-dir", PROJECT / "data/llm2_training",
           "--adapter", PROJECT / "models/llm2-detailed-v3-adapter",
           "--output-dir", PROJECT / "reports/llm2-detailed-v3-evaluation",
           "--test-limit", TEST_LIMIT, "--generation-limit", GENERATION_LIMIT,
           "--challenge-limit", 0, "--rag-ablation-limit", 0,
           "--max-new-tokens", 2048, "--max-generation-context", 8192)
report = json.loads((PROJECT / "reports/llm2-detailed-v3-evaluation/evaluation_report.json").read_text())
print("Adapter minus base held-out NLL:", report["adapter_minus_base_nll"])
print("Incomplete generations:", report["incomplete_generations"])


## Export the new adapter, evaluation, and data provenance


In [ ]:
export_path = Path("/kaggle/working/sms-ac-llm2-detailed-v3.zip")
adapter = PROJECT / "models/llm2-detailed-v3-adapter"
manifest = json.loads((adapter / "model_manifest.json").read_text())
assert manifest["training_completed"] and manifest["evaluation_status"] == "paired_heldout_complete"
files = [p for p in adapter.iterdir() if p.is_file()]
files += [p for p in (PROJECT / "reports/llm2-detailed-v3-evaluation").rglob("*") if p.is_file()]
files += [PROJECT / "data/llm2_training/conversion_manifest.json"]
with zipfile.ZipFile(export_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(files)):
        archive.write(path, path.relative_to(PROJECT))
print("Saved:", export_path, "MiB:", round(export_path.stat().st_size / 1024**2, 1))
from IPython.display import FileLink, display
display(FileLink(str(export_path)))
